Quiz 53 — Customer Support SLA Full Analysis  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(160)
n        = 150
priority = np.random.choice([1,2,3], n, p=[0.2,0.5,0.3])
np.random.seed(160)
r1 = np.random.exponential(4,  n)
r2 = np.random.exponential(18, n)
r3 = np.random.exponential(48, n)
res_hours = np.where(priority==1, r1, np.where(priority==2, r2, r3))
agent_id  = np.random.randint(1, 8, n)
week      = np.repeat([1,2,3,4], [40,38,37,35])

# ── 1. SLA breach flags ───────────────────────────────────────────────────────
sla_limit = np.where(priority==1, 4, np.where(priority==2, 24, 72))
breached  = (res_hours > sla_limit).astype(int)
print(f"Overall breach rate: {breached.mean()*100:.1f}%")

# ── 2. Breach rate per priority and per week ──────────────────────────────────
print("\nBreach rate by priority:")
for p in [1, 2, 3]:
    m = priority == p
    print(f"  P{p}: {breached[m].mean()*100:.1f}%  ({m.sum()} tickets)")

print("\nBreach rate by week:")
week_breach = []
for wk in [1,2,3,4]:
    m = week == wk
    br = breached[m].mean() * 100
    week_breach.append(br)
    print(f"  Week {wk}: {br:.1f}%")

# ── 3. Per-agent stats ────────────────────────────────────────────────────────
print("\nAgent analysis:")
for ag in range(1, 8):
    m = agent_id == ag
    if m.sum() == 0:
        continue
    br    = breached[m].mean() * 100
    flag  = " ← Needs Review" if br > 40 else ""
    print(f"  Agent {ag}: {m.sum():3d} tickets  breach={br:5.1f}%  "
          f"mean_res={res_hours[m].mean():.1f}h{flag}")

# ── 4. Percentile distribution per priority ───────────────────────────────────
print("\nResolution time percentiles:")
for p in [1, 2, 3]:
    m = priority == p
    pcts = np.percentile(res_hours[m], [25,50,75,90])
    print(f"  P{p}: P25={pcts[0]:.1f}h  P50={pcts[1]:.1f}h  "
          f"P75={pcts[2]:.1f}h  P90={pcts[3]:.1f}h")

# ── 5. Bar chart ──────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.bar([1,2,3,4], week_breach, color="steelblue", edgecolor="white")
plt.axhline(20, color="red", linestyle="--", linewidth=2, label="20% target")
plt.title("SLA Breach Rate by Week")
plt.xlabel("Week")
plt.ylabel("Breach Rate (%)")
plt.ylim(0, max(week_breach) + 10)
plt.legend()
plt.tight_layout()
plt.savefig("hard_13_sla_weekly.png", dpi=100)
plt.show()